# Day 048 — Exercise 2: encode_categoricals

**What you'll build:** `encode_categoricals(df, cat_cols=None, drop_first=False) -> pd.DataFrame` — one-hot encode string columns with `pd.get_dummies` and convert the resulting bool columns to int so sklearn's estimators see numbers.

**Why it matters:** Most ML models can't handle string data — only numbers. One-hot encoding converts a column like `neighborhood` with values `'downtown'`, `'suburb'`, `'rural'` into three binary columns. `drop_first=True` drops one column to avoid multicollinearity (the dummy variable trap) when using linear models.

## Provided: Setup + prepare_features + split_data

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })


def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }

## Your Implementation

In [ ]:
def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """
    One-hot encode categorical (object dtype) columns.

    Args:
        df:         input DataFrame
        cat_cols:   list of columns to encode; if None, auto-detect
                    all columns with dtype 'object'
        drop_first: if True, drop one dummy per category (avoids
                    multicollinearity in linear models)
    Returns:
        New DataFrame with cat_cols replaced by integer dummy columns.
        Original numeric columns are unchanged.
    """
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    # TODO: encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # TODO: # pandas 2.x returns bool dtype for dummies — convert to int
    # TODO: bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    # TODO: for c in bool_cols:
    #     encoded[c] = encoded[c].astype(int)
    # TODO: return encoded
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = make_regression_data(100)

    # Check 1: defined, returns DataFrame
    try:
        assert 'encode_categoricals' in globals()
        encoded = encode_categoricals(df)
        assert isinstance(encoded, pd.DataFrame), \
            f'expected DataFrame, got {type(encoded).__name__}'
        passed += 1; print(f'\u2705 Check 1: encode_categoricals returns DataFrame ({encoded.shape[1]} cols)')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: original cat column removed
    try:
        assert 'neighborhood' not in encoded.columns, \
            'neighborhood should be gone after encoding'
        passed += 1; print('\u2705 Check 2: original categorical column removed')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: dummy columns created for each category value
    try:
        dummy_cols = [c for c in encoded.columns if c.startswith('neighborhood_')]
        assert len(dummy_cols) == 3, \
            f'expected 3 neighborhood_ columns, got {len(dummy_cols)}: {dummy_cols}'
        passed += 1; print(f'\u2705 Check 3: 3 dummy columns created: {dummy_cols}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: dummy columns are numeric (int), not bool
    try:
        for c in [col for col in encoded.columns if col.startswith('neighborhood_')]:
            assert encoded[c].dtype != bool, \
                f'{c} should not be bool; convert to int'
            assert pd.api.types.is_numeric_dtype(encoded[c]), \
                f'{c} should be numeric'
        passed += 1; print('\u2705 Check 4: dummy columns are numeric (int), not bool')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: drop_first=True produces one fewer column
    try:
        enc_drop = encode_categoricals(df, drop_first=True)
        assert enc_drop.shape[1] < encoded.shape[1], \
            f'drop_first=True should reduce columns ({enc_drop.shape[1]} < {encoded.shape[1]})'
        passed += 1; print(f'\u2705 Check 5: drop_first=True → {enc_drop.shape[1]} cols (vs {encoded.shape[1]})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode categorical columns with pd.get_dummies."""
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # pandas 2.x returns bool dtype for dummy columns; convert to int
    bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    for c in bool_cols:
        encoded[c] = encoded[c].astype(int)
    return encoded
```

</details>